In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from memorywrap import MemoryWrapLayer
from resnet import MemoryResNet18 as MemoryResNet
from data import train_loader, test_loader, mem_loader

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import cv2
import os

from data import train_loader, test_loader, mem_loader

# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = "mps"
model = MemoryResNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
criterion = nn.CrossEntropyLoss()

os.makedirs('vis', exist_ok=True)


# ---------------------------------------------------------------------------
def denormalize(tensor, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    """Undo dataset normalisation for display. tensor: (3, H, W)"""
    t = tensor.clone().cpu().float()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1).permute(1, 2, 0).numpy()


def overlay_heatmap(img_np, map_np, orig_h, orig_w):
    """Resize map to image size and blend as jet heatmap."""
    m = cv2.resize(map_np, (orig_w, orig_h))
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    heatmap = plt.cm.jet(m)[..., :3]          # (H, W, 3)
    return 0.5 * img_np + 0.5 * heatmap


# ---------------------------------------------------------------------------
def visualize_batch(x, ss, query_maps, memory_maps, att_weights, labels, epoch, batch_idx,
                    n_examples=4, top_k=3):
    """
    att_weights: (B, N)  attention weights from MemoryWrapLayer
    top_k:       how many attended memory images to show per query
    """
    B = min(n_examples, x.shape[0])
    N_total = memory_maps.shape[1]
    orig_h, orig_w = x.shape[2], x.shape[3]

    ss_grid = ss.view(B, N_total, 3, orig_h, orig_w)

    # columns: query | query+heatmap | then top_k x (mem | mem+heatmap)
    n_cols = 2 + 2 * top_k
    fig = plt.figure(figsize=(3 * n_cols, 3 * B))
    gs  = gridspec.GridSpec(B, n_cols, figure=fig, hspace=0.05, wspace=0.05)

    for b in range(B):
        img_q = denormalize(x[b])
        qmap  = query_maps[b].detach().cpu().numpy()

        # --- pick top_k attended memory indices for this query ---
        weights   = att_weights[b].detach().cpu()        # (N,)
        top_idx   = weights.topk(top_k).indices.numpy()  # indices of top attended memories

        # --- query image ---
        ax = fig.add_subplot(gs[b, 0])
        ax.imshow(img_q)
        if b == 0:
            ax.set_title('Query', fontsize=8)
        ax.axis('off')
        ax.set_ylabel(f'label {labels[b].item()}', fontsize=7)

        # --- query + importance map ---
        ax = fig.add_subplot(gs[b, 1])
        ax.imshow(overlay_heatmap(img_q, qmap, orig_h, orig_w))
        if b == 0:
            ax.set_title('Query\nimportance', fontsize=8)
        ax.axis('off')

        # --- only the top attended memory images ---
        for plot_n, mem_n in enumerate(top_idx):
            img_m = denormalize(ss_grid[b, mem_n])
            mmap  = memory_maps[b, mem_n].detach().cpu().numpy()
            w     = weights[mem_n].item()
            col   = 2 + 2 * plot_n

            ax = fig.add_subplot(gs[b, col])
            ax.imshow(img_m)
            if b == 0:
                ax.set_title(f'Mem {mem_n}\nattn {w:.3f}', fontsize=8)
            ax.axis('off')

            ax = fig.add_subplot(gs[b, col + 1])
            ax.imshow(overlay_heatmap(img_m, mmap, orig_h, orig_w))
            if b == 0:
                ax.set_title(f'Mem {mem_n}\ncorr map', fontsize=8)
            ax.axis('off')

    fig.suptitle(f'Epoch {epoch} — batch {batch_idx}', fontsize=9, y=1.01)
    path = f'vis/epoch{epoch:03d}_batch{batch_idx:04d}.png'
    plt.savefig(path, bbox_inches='tight', dpi=120)
    plt.close(fig)
    print(f'  saved {path}')


# ---------------------------------------------------------------------------
def train_epoch(model, train_loader, mem_loader, optimizer, criterion, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    mem_iter = iter(mem_loader)
    for batch_idx, (x, y) in enumerate(train_loader):
        # --- refresh mem iterator if exhausted ---
        try:
            mem_x, _ = next(mem_iter)
        except StopIteration:
            mem_iter = iter(mem_loader)
            mem_x, _ = next(mem_iter)

        x, y       = x.to(device), y.to(device)
        mem_x      = mem_x.to(device)

        # mem_x is (M, 3, H, W); repeat to align with batch:
        # MemoryWrap expects ss as (B*N, 3, H, W) where every query
        # in the batch sees the same memory set
        B = x.shape[0]
        N = mem_x.shape[0]
        ss = mem_x.unsqueeze(0).expand(B, -1, -1, -1, -1)  # (B, N, 3, H, W)
        ss = ss.reshape(B * N, *mem_x.shape[1:])            # (B*N, 3, H, W)

        optimizer.zero_grad()
        (logits, _att), memory_maps, query_maps = model(x, ss, return_weights=True)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += logits.argmax(1).eq(y).sum().item()
        total      += B

        if batch_idx % 100 == 0:
            print(f'  [train] epoch {epoch} step {batch_idx}/{len(train_loader)} '
                  f'loss {loss.item():.4f} acc {100.*correct/total:.1f}%')

    return total_loss / len(train_loader), 100. * correct / total


# ---------------------------------------------------------------------------
def test_epoch(model, test_loader, mem_loader, criterion, device, epoch,
               vis_every=50, n_vis_examples=4):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    mem_iter = iter(mem_loader)
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(test_loader):
            try:
                mem_x, _ = next(mem_iter)
            except StopIteration:
                mem_iter = iter(mem_loader)
                mem_x, _ = next(mem_iter)

            x, y  = x.to(device), y.to(device)
            mem_x = mem_x.to(device)

            B = x.shape[0]
            N = mem_x.shape[0]
            ss = mem_x.unsqueeze(0).expand(B, -1, -1, -1, -1)
            ss = ss.reshape(B * N, *mem_x.shape[1:])

            (logits, att_weights), memory_maps, query_maps = model(x, ss, return_weights=True)
            loss = criterion(logits, y)

            total_loss += loss.item()
            correct    += logits.argmax(1).eq(y).sum().item()
            total      += B

            # --- visualise every vis_every batches ---
            if batch_idx % vis_every == 0:
                visualize_batch(
                    x=x[:n_vis_examples],
                    ss=ss.view(B, N, *x.shape[1:])[:n_vis_examples].reshape(-1, *x.shape[1:]),
                    query_maps=query_maps[:n_vis_examples],
                    memory_maps=memory_maps[:n_vis_examples],
                    att_weights=att_weights[:n_vis_examples],
                    labels=y[:n_vis_examples],
                    epoch=epoch,
                    batch_idx=batch_idx,
                )

    acc = 100. * correct / total
    print(f'  [test]  epoch {epoch} loss {total_loss/len(test_loader):.4f} acc {acc:.1f}%')
    return total_loss / len(test_loader), acc


# ---------------------------------------------------------------------------
def run(epochs=200):
    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        print(f'\n=== Epoch {epoch}/{epochs} ===')

        train_loss, train_acc = train_epoch(
            model, train_loader, mem_loader, optimizer, criterion, device, epoch)

        test_loss, test_acc = test_epoch(
            model, test_loader, mem_loader, criterion, device, epoch)

        scheduler.step()

        print(f'  train loss {train_loss:.4f}  acc {train_acc:.1f}% | '
              f'test loss {test_loss:.4f}  acc {test_acc:.1f}%')

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'  ** new best: {best_acc:.1f}% — saved **')

In [3]:
run()


=== Epoch 1/200 ===
  [train] epoch 1 step 0/3125 loss 2.3088 acc 0.0%
  [train] epoch 1 step 100/3125 loss 2.3857 acc 13.9%
  [train] epoch 1 step 200/3125 loss 1.9396 acc 14.7%
  [train] epoch 1 step 300/3125 loss 2.4977 acc 14.7%
  [train] epoch 1 step 400/3125 loss 1.8368 acc 15.5%


KeyboardInterrupt: 